# 5 Selecting files: globbing, brace expansion, and find

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part II — Pipelines and text extraction</span>
    <span class="bp-meta">Notebook&nbsp;5</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    Referring to many files at once (by pattern, by generation, or by searching
    the tree), and then acting on the whole set.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v1.0.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate. data/ is
# read-only; anything we create goes into a fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"

## What this notebook is about

Notebook 2 got you *to* a directory; Notebook 4 gave you pipes. But a real project
has *hundreds* of files, and you are not going to name them one at a time. So this
notebook answers a single question: **how do you refer to many files at once, and
act on the whole set?**

There are three ways to select, and one way to act, and they share one idea:

- **glob**: match files that already exist, *here* (`*.xyz`);
- **brace expansion**: *generate* names and sequences, exist or not (`run_{1..10}`);
- **`find`**: search the whole *tree* by criteria (every `.xyz`, anywhere);
- **`xargs`**: take any of those lists and *run a command* over all of it.

The files are the real `.xyz` trajectories and simulation logs from before; as
ever, **selecting them needs no physics.** Let's take the four in turn, simplest
first.

## A. Globbing — match files that are here

A **glob** is a pattern the shell expands into a list of matching filenames. The
pieces:

<div class="bp-card">
  <span class="bp-card-cmd">Globs</span> — <span class="bp-card-job">wildcards the shell expands into matching filenames (shell syntax, not a command).</span>
  <table>
    <tr><td>*</td><td>any run of characters (including none)</td></tr>
    <tr><td>?</td><td>exactly one character</td></tr>
    <tr><td>[abc]</td><td>one character from the set</td></tr>
    <tr><td>[1-5]</td><td>one character from the range</td></tr>
    <tr><td>[!abc]</td><td>one character <b>not</b> in the set</td></tr>
  </table>
</div>

`*` is the workhorse. Here it matches every `.xyz` one level down in `data/`:

In [2]:
ls data/*/*.xyz

data/results/pt-slab.xyz		 data/trajectories/lj38-relaxed.xyz


data/trajectories/lj38-optimization.xyz


The load-bearing idea is worth stopping on, because it explains
half of bash's surprises. **The shell expands the glob *before the command runs*.**
`ls` never sees `*.xyz`; it sees the three real filenames the shell already
substituted. You can watch that substitution directly with `echo`, which just
prints whatever it is handed:

In [3]:
echo data/*/*.xyz

data/results/pt-slab.xyz data/trajectories/lj38-optimization.xyz data/trajectories/lj38-relaxed.xyz


The other wildcards need a batch of similarly-named files to show off, so let us
make one in `scratch/` (a stand-in for a project's worth of numbered runs):

In [4]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/runs
touch scratch/runs/run_{1,2,3,10}.log scratch/runs/run_{1,2,3}.xyz

In [5]:
ls scratch/runs

run_1.log  run_1.xyz  run_10.log  run_2.log  run_2.xyz	run_3.log  run_3.xyz


`?` matches exactly one character, so `run_?.log` catches `run_1` through `run_9`
but **not** `run_10`, which has two digits:

In [6]:
echo scratch/runs/run_?.log

scratch/runs/run_1.log scratch/runs/run_2.log scratch/runs/run_3.log


A bracket matches one character from a set or range:

In [7]:
ls scratch/runs/run_[1-2].log

scratch/runs/run_1.log	scratch/runs/run_2.log


Two gotchas worth carrying forward. First, a glob that matches **nothing** is, by
bash's default, passed through to the command *literally*: the pattern itself,
unexpanded:

In [8]:
echo scratch/runs/*.dat

scratch/runs/*.dat


There is no `.dat` file, so `*.dat` arrived verbatim. (Bash has a `nullglob`
option that makes a non-match expand to nothing instead; just know it exists.)
Second, globs **skip dotfiles**: `data/*` quietly passed over the hidden
`.dataset-notes` from Notebook 2:

In [9]:
echo data/*

data/README.md data/inputs data/logs data/results data/scaling data/trajectories


## B. Brace expansion — generate names and sequences

Globbing *matches* files that exist. **Brace expansion** is its complement: it
*generates* strings whether or not any file exists, and the shell does it even
earlier than globbing.

<div class="bp-card">
  <span class="bp-card-cmd">Brace expansion</span> — <span class="bp-card-job">generate lists and sequences of strings (shell syntax). Runs before globbing; the files need not exist.</span>
  <table>
    <tr><td>{a,b,c}</td><td>each item in turn (no spaces inside!)</td></tr>
    <tr><td>{1..10}</td><td>a numeric range</td></tr>
    <tr><td>{1..10..2}</td><td>a range with a step</td></tr>
    <tr><td>pre{a,b}post</td><td>a shared prefix/suffix is distributed over each item</td></tr>
    <tr><td>file{,.bak}</td><td>the empty item gives <code>file</code> and <code>file.bak</code>: the backup idiom</td></tr>
  </table>
</div>

Watch the generation with `echo`:

In [10]:
echo run_{1..5}

run_1 run_2 run_3 run_4 run_5


In [11]:
echo frame_{0..10..2}.xyz

frame_0.xyz frame_2.xyz frame_4.xyz frame_6.xyz frame_8.xyz frame_10.xyz


Because the strings are generated regardless of what exists, braces are how you
*create* a structured set in one stroke. `mkdir -p run_{1..5}` makes five
directories at once:

In [12]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [13]:
mkdir -p scratch/run_{1..5}

In [14]:
ls scratch

run_1  run_2  run_3  run_4  run_5


And the `file{,.bak}` idiom expands `cp config.txt{,.bak}` into
`cp config.txt config.txt.bak`: an instant backup in nine keystrokes:

In [15]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'cutoff = 300\n' > scratch/config.txt

In [16]:
cp scratch/config.txt{,.bak}

In [17]:
ls scratch

config.txt  config.txt.bak


## C. `find` — search the whole tree by criteria

Globs and braces work on names in *one* directory. `find` makes the leap from
"names here" to "**criteria anywhere in the tree**": walk a directory and
everything under it, keep the files that match a set of tests, and (optionally)
run a command on each.

```{command-card} find
```

The shape is `find <where> <tests> <action>`. Start simple: every `.xyz`
underneath `data/`, wherever it lives:

In [18]:
find data -name "*.xyz"

data/trajectories/lj38-optimization.xyz


data/trajectories/lj38-relaxed.xyz


data/results/pt-slab.xyz


It reached into both `trajectories/` and `results/`, something no single glob can
do. **Quote the pattern** (`"*.xyz"`): that is the payoff of §A. If you leave it
bare, the *shell* expands `*.xyz` before `find` ever runs, and `find` gets the
wrong thing. Quoting hands the pattern to `find` intact.

Tests combine. Only directories:

In [19]:
find data -type d

data


data/inputs


data/logs


data/scaling


data/trajectories


data/results


Keep it shallow with `-maxdepth`, and select by size — here, files bigger than 50
kilobytes, which finds the one large trajectory:

In [20]:
find data -size +50k

data/trajectories/lj38-optimization.xyz


Tests can be negated with `!` and combined with `-o` (or). This finds every `.xyz`
that is *not* under `results/`:

In [21]:
find data -name "*.xyz" ! -path "*/results/*"

data/trajectories/lj38-optimization.xyz


data/trajectories/lj38-relaxed.xyz


:::{admonition} ⚠ find can delete at scale — dry-run first, always
:class: danger
`find` has a `-delete` action and an `-exec rm` action, and they are the Notebook
3 `rm` footgun multiplied by *every file in the tree*. One wrong test (a `-name`
that is broader than you meant, a stray `-o`) and an entire subtree is gone, with
no undo and no trash. The habit that saves you is simple and non-negotiable:
**run the `find` with `-print` first and read the list**, every time. Only once you
have *seen* exactly what it matches do you swap `-print` for `-delete`. Never pipe
a `find` you have not eyeballed into anything that removes files.
:::

## D. `xargs` — act on a list

`find` *selects* files; `xargs` *acts* on them. It reads a list of items on
standard input and turns them into arguments for a command, which means it is, at
heart, just a pipe (Notebook 4): a list flows in, command lines come out.

```{command-card} xargs
```

The classic pairing pulls the first line (the atom count) out of *every*
trajectory `find` can reach:

In [22]:
find data -name "*.xyz" -print0 | xargs -0 head -n 1

==> data/trajectories/lj38-optimization.xyz <==


      38


==> data/trajectories/lj38-relaxed.xyz <==


      38


==> data/results/pt-slab.xyz <==


180


Two details carry the safety lesson. `find … -print0` separates names with an
invisible NUL character instead of whitespace, and `xargs -0` reads them the same
way, so a filename with a space or newline in it cannot be split in two and
mangled. Make `-print0 | xargs -0` your default pairing.

`find` also has its own built-in way to act, `-exec`, which needs no pipe at all.
The `{}` stands for each match and the trailing `+` batches them efficiently:

In [23]:
find data -name "*.xyz" -exec head -n 1 {} +

==> data/trajectories/lj38-optimization.xyz <==


      38


==> data/trajectories/lj38-relaxed.xyz <==


      38


==> data/results/pt-slab.xyz <==


180


Same result, two routes: lead with `xargs` when you want a general "list →
arguments" tool that composes with *any* producer, and reach for `-exec` when the
list is coming from `find` anyway. (One more thing `xargs` can do, for later: it
will run those command lines *in parallel* with `-P`. We come back to that when we
talk about speed in Part IV.)

## Exercises

A fuller set this time: selection is a theme worth practising from several angles.
Reads over `data/` are fine; anything that creates or deletes works in a fresh
`scratch/`, set up at the top of each exercise so reruns are identical.

### Warm-up 1 (worked) — Globs

Match files with globs and watch a non-match pass through literally.

In [24]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/runs
touch scratch/runs/sweep_{1,2,3,4,5}.log scratch/runs/sweep_{1,2,3}.xyz

In [25]:
# (solution hidden on the public site)


scratch/runs/sweep_1.xyz  scratch/runs/sweep_2.xyz  scratch/runs/sweep_3.xyz


scratch/runs/sweep_1.log  scratch/runs/sweep_3.log  scratch/runs/sweep_5.log


scratch/runs/sweep_2.log  scratch/runs/sweep_4.log


scratch/runs/*.dat


In [26]:
check '[ "$(ls scratch/runs/*.xyz | wc -l)" -eq 3 ] && [ "$(ls scratch/runs/sweep_?.log | wc -l)" -eq 5 ] && [ "$(echo scratch/runs/*.dat)" = "scratch/runs/*.dat" ]' \
      "the .xyz glob matched 3, sweep_?.log matched 5, and the non-match passed through literally"

✓ the .xyz glob matched 3, sweep_?.log matched 5, and the non-match passed through literally


### Warm-up 2 (your turn) — Brace expansion

In `scratch/`, make five run directories `run_1` … `run_5` in one `mkdir -p`, and
make a backup of a file with the `file{,.bak}` idiom.

In [27]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'seed = 1\n' > scratch/input.txt

In [28]:
# (solution hidden on the public site)


input.txt  input.txt.bak  run_1  run_2	run_3  run_4  run_5


In [29]:
check '[ -d scratch/run_1 ] && [ -d scratch/run_5 ] && [ "$(ls -d scratch/run_* | wc -l)" -eq 5 ] && [ -f scratch/input.txt.bak ]' \
      "five run_ directories exist and the .bak backup was made"

✓ five run_ directories exist and the .bak backup was made


### Applied 1 (your turn) — find by criteria

Using `find` on `data/` (read-only): find every `.inp` input file (quote the
pattern!), and separately every directory. Then list only what is *directly*
inside `data/` with `-maxdepth 1`.

In [30]:
cd "$ROOT"

In [31]:
# (solution hidden on the public site)


data/inputs/geo-opt.inp


data/inputs/production-md.inp


data


data/inputs


data/logs


...

data/inputs


data/logs


data/scaling


data/trajectories


data/results


In [32]:
check '[ "$(find data -name "*.inp" | wc -l)" -eq 2 ] && [ "$(find data -type d | wc -l)" -eq 6 ]' \
      "find matched both .inp inputs and all six directories"

✓ find matched both .inp inputs and all six directories


### Applied 2 (worked) — find | xargs

Pull the first line out of every trajectory in `data/`, two ways: piped through
`xargs`, and with `find`'s own `-exec`.

In [33]:
cd "$ROOT"

In [34]:
# (solution hidden on the public site)


==> data/trajectories/lj38-optimization.xyz <==


      38


==> data/trajectories/lj38-relaxed.xyz <==


      38


==> data/results/pt-slab.xyz <==


180


In [35]:
# (solution hidden on the public site)


==> data/trajectories/lj38-optimization.xyz <==


      38


==> data/trajectories/lj38-relaxed.xyz <==


      38


==> data/results/pt-slab.xyz <==


180


In [36]:
check '[ "$(find data -name "*.xyz" | wc -l)" -eq 3 ] && [ -n "$(find data -name "*.xyz" -print0 | xargs -0 head -n 1)" ]' \
      "all three trajectories were found and their first lines pulled"

✓ all three trajectories were found and their first lines pulled


### Composite — putting it together (capstone)

Build a small sorted archive in `scratch/`, exercising the whole notebook at once.
Make three brace-generated bins `scratch/sorted/{R,S,M}`, then use `find` + `xargs`
to copy every `.xyz` from `data/` into the `R` bin, and verify with `find`.

In [37]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [38]:
# (solution hidden on the public site)


scratch/sorted/R/pt-slab.xyz


scratch/sorted/R/lj38-optimization.xyz


scratch/sorted/R/lj38-relaxed.xyz


In [39]:
check '[ -d scratch/sorted/R ] && [ -d scratch/sorted/S ] && [ -d scratch/sorted/M ] && [ "$(find scratch/sorted -name "*.xyz" | wc -l)" -eq 3 ]' \
      "the three brace-made bins exist and all three .xyz were copied into R/"

✓ the three brace-made bins exist and all three .xyz were copied into R/


### Optional stretch (your turn) — Boolean find

Compose a single `find` with combined tests: every `.xyz` under `data/` that is
**not** in `results/` **and** is larger than 1 kilobyte, then count them with
`xargs`. (Aside: `xargs` could run the action in parallel with `-P`, a thread we
pick up in Part IV.)

In [40]:
cd "$ROOT"

In [41]:
# (solution hidden on the public site)


-rw-r--r-- 1 runner runner 155K Apr  1  2023 data/trajectories/lj38-optimization.xyz


-rw-r--r-- 1 runner runner 2.5K Apr  1  2023 data/trajectories/lj38-relaxed.xyz


In [42]:
check '[ "$(find data -name "*.xyz" ! -path "*/results/*" -size +1k | wc -l)" -eq 2 ]' \
      "the boolean find selected exactly the two large trajectories outside results/"

✓ the boolean find selected exactly the two large trajectories outside results/


## Outlook

You can now select files at scale (by pattern, by generation, by searching the
tree) and act on the whole set at once. Next (Notebook 6): instead of finding
*files*, you will find matching **text inside** them with `grep`, and meet
**regular expressions**: the pattern language that `sed`, and the Vim
search-and-replace from Notebook 9, all share.

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to run everything yourself; nothing to install. The published
    notebooks ship <b>without worked solutions</b>; if you would like the
    reference solutions (to teach from or to check your own work), get in
    touch: <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>